In [ ]:
import pandas as pd
from difflib import SequenceMatcher

# Load dataset
df = pd.read_csv(
    "/content/clean_jobs_descriptions_combined (1).csv"
)

print("Dataset loaded successfully!")
print("Rows:", len(df))
print("Columns:")
print(df.columns.tolist())

Dataset loaded successfully!
Rows: 8785
Columns:
['Job Id', 'Job Title', 'Company', 'location', 'Job Description', 'Experience', 'Qualifications', 'Salary Range', 'Work Type', 'skills', 'clean_description', 'extracted_skills', 'categories']


In [ ]:

COMMON_SKILLS = [
    "Python",
    "SQL",
    "Java",
    "JavaScript",
    "TypeScript",
    "HTML",
    "CSS",
    "C++",
    "C#",
    "Ruby",
    "PHP",
    "React",
    "Angular",
    "Vue",
    "Node.js",
    "Machine Learning",
    "Deep Learning",
    "NLP",
    "TensorFlow",
    "PyTorch",
    "Scikit-learn",
    "Pandas",
    "NumPy",
    "Matplotlib",
    "AWS",
    "Azure",
    "Google Cloud",
    "GCP",
    "Docker",
    "Kubernetes",
    "Git",
    "GitHub",
    "Linux",
    "Excel",
    "Power BI",
    "Tableau",
    "Spark",
    "Hadoop",
    "MongoDB",
    "MySQL",
    "PostgreSQL",
    "Oracle",
    "Django",
    "Flask",
    "FastAPI",
    "REST API",
    "API",
    "Data Analysis",
    "Data Visualization",
    "Statistics",
    "Data Science"
]

In [ ]:
import ast
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
import re
import numpy as np

# Map lowercase common skills to their original casing for consistent matching
common_skills_map = {skill.lower(): skill for skill in COMMON_SKILLS}

# Ensure 'extracted_skills' column is a list of strings
# Assuming 'extracted_skills' might be stored as a string representation of a list.
# Handle potential NaN values by converting them to empty lists.
def clean_skill_entry(skill_str):
    if not isinstance(skill_str, str):
        return []
    skills = []
    # Split by comma, then process each part
    for part in skill_str.split(','):
        s_stripped = part.strip()
        # Remove anything in parentheses, then strip again
        cleaned_s = re.sub(r'\(.*\)', '', s_stripped).strip()

        if cleaned_s:
            # Normalize to lowercase for lookup and then map back to original casing
            cleaned_s_lower = cleaned_s.lower()
            if cleaned_s_lower in common_skills_map:
                skills.append(common_skills_map[cleaned_s_lower])
    return skills

df['extracted_skills_cleaned'] = df['extracted_skills'].apply(clean_skill_entry)

# Diagnostic: Print first few cleaned skill lists
print("\nFirst 5 entries of df['extracted_skills_cleaned']:")
print(df['extracted_skills_cleaned'].head().tolist())

# Initialize and fit MultiLabelBinarizer with all common skills
mlb = MultiLabelBinarizer()
mlb.fit([COMMON_SKILLS]) # Fit on a list containing all possible skills

# Transform the cleaned skills into a multi-hot encoded format
y_all = mlb.transform(df['extracted_skills_cleaned'])

# Diagnostic: Print statistics about y_all
print("\nShape of y_all:", y_all.shape)
print("Number of samples with at least one recognized skill:", np.sum(np.sum(y_all, axis=1) > 0))
print("Percentage of samples with at least one recognized skill: {:.2f}%".format(np.sum(np.sum(y_all, axis=1) > 0) / y_all.shape[0] * 100))



First 5 entries of df['extracted_skills_cleaned']:
[[], ['JavaScript', 'HTML', 'CSS', 'HTML', 'CSS', 'React', 'Angular'], [], [], []]

Shape of y_all: (8785, 51)
Number of samples with at least one recognized skill: 1101
Percentage of samples with at least one recognized skill: 12.53%


In [ ]:
# Assuming 'clean_description' is the feature column (X)
X_train, X_test, y_train, y_test = train_test_split(df['clean_description'], y_all, test_size=0.2, random_state=42)

# Create a dummy y_pred. For example, a slightly modified version of y_test
# This simulates some correct and incorrect predictions for demonstration.
y_pred = y_test.copy()
# Introduce some 'errors' for demonstration if needed
# For simplicity, we'll just use a direct copy for this placeholder

print("mlb, X_test, y_test, and y_pred are now defined.")

mlb, X_test, y_test, and y_pred are now defined.


In [ ]:

# Create actual and predicted skill lists

actual_skill_lists = mlb.inverse_transform(y_test)

predicted_skill_lists = mlb.inverse_transform(y_pred)

print("Skill lists created successfully!")

Skill lists created successfully!


In [ ]:
print("Actual example:")
print(actual_skill_lists[0])

print("\nPredicted example:")
print(predicted_skill_lists[0])

Actual example:
()

Predicted example:
()


In [ ]:
SYNONYMS = {
    "ML": "Machine Learning",
    "AI": "Artificial Intelligence",
    "JS": "JavaScript",
    "Postgres": "PostgreSQL",
    "PowerBI": "Power BI",
    "sklearn": "Scikit-learn"
}

print("Synonym dictionary created!")
print(SYNONYMS)

Synonym dictionary created!
{'ML': 'Machine Learning', 'AI': 'Artificial Intelligence', 'JS': 'JavaScript', 'Postgres': 'PostgreSQL', 'PowerBI': 'Power BI', 'sklearn': 'Scikit-learn'}


In [ ]:
#create normalize function
def normalize(text):

    return str(text).lower().strip()


print(normalize("  Python  "))
print(normalize("Machine Learning"))

python
machine learning


In [ ]:
def is_synonym(actual, predicted):

    actual = normalize(actual)

    predicted = normalize(predicted)

    for short_form, full_form in SYNONYMS.items():

        if (
            predicted == normalize(short_form)
            and
            actual == normalize(full_form)
        ):

            return True

    return False


print(
    is_synonym(
        "Machine Learning",
        "ML"
    )
)

print(
    is_synonym(
        "Python",
        "Java"
    )
)

True
False


In [ ]:
#partial entities
def is_partial(actual, predicted):

    actual = normalize(actual)

    predicted = normalize(predicted)

    if actual == predicted:

        return False

    if (
        actual in predicted
        or
        predicted in actual
    ):

        return True

    return False


print(
    is_partial(
        "Python",
        "Python 3"
    )
)

print(
    is_partial(
        "Python",
        "Java"
    )
)

True
False


In [ ]:
def is_spelling_error(actual, predicted):

    actual = normalize(actual)

    predicted = normalize(predicted)

    if actual == predicted:

        return False

    similarity = SequenceMatcher(
        None,
        actual,
        predicted
    ).ratio()

    return similarity >= 0.80

In [ ]:
print(
    is_spelling_error(
        "Python",
        "Pythn"
    )
)

print(
    is_spelling_error(
        "Python",
        "Java"
    )
)

True
False


In [ ]:
error_rows = []

for i in range(len(actual_skill_lists)):

    actual_skills = set(
        actual_skill_lists[i]
    )

    predicted_skills = set(
        predicted_skill_lists[i]
    )

    text = X_test.iloc[i]

In [17]:
#detect false positive
# FALSE POSITIVE
# Model predicted a skill that was not in actual skills

for predicted in predicted_skills - actual_skills:

    error_type = "False Positive"

    error_rows.append({
        "Text": text,
        "Actual": ", ".join(
            sorted(actual_skills)
        ),
        "Predicted": predicted,
        "Error": error_type
    })

In [18]:
# FALSE NEGATIVE

for actual in actual_skills -predicted_skills:

        error_rows.append({
            "Text": text,
            "Actual": actual,
            "Predicted": ", ".join(
                sorted(predicted_skills)
            ),
            "Error": "False Negative"
        })


print("Error analysis completed!")
print("Total errors:", len(error_rows))

Error analysis completed!
Total errors: 0


In [19]:
error_df = pd.DataFrame(
    error_rows
)

print("\nError Analysis Table")
print("====================")

print(
    error_df.head(20).to_string(
        index=False
    )
)


Error Analysis Table
Empty DataFrame
Columns: []
Index: []


In [20]:
partial_rows = []

for _, row in error_df.iterrows():

    actual = row["Actual"]
    predicted = row["Predicted"]

    if is_partial(
        actual,
        predicted
    ):

        partial_rows.append({
            "Text": row["Text"],
            "Actual": actual,
            "Predicted": predicted,
            "Error": "Partial Entity"
        })


print(
    "Partial Entity cases:",
    len(partial_rows)
)

Partial Entity cases: 0


In [21]:
#adding synonymns analysis
synonym_rows = []

for _, row in error_df.iterrows():

    actual = row["Actual"]
    predicted = row["Predicted"]

    if is_synonym(
        actual,
        predicted
    ):

        synonym_rows.append({
            "Text": row["Text"],
            "Actual": actual,
            "Predicted": predicted,
            "Error": "Synonym Problem"
        })


print(
    "Synonym Problem cases:",
    len(synonym_rows)
)

Synonym Problem cases: 0


In [22]:
spelling_rows = []

for _, row in error_df.iterrows():

    actual = row["Actual"]
    predicted = row["Predicted"]

    if is_spelling_error(
        actual,
        predicted
    ):

        spelling_rows.append({
            "Text": row["Text"],
            "Actual": actual,
            "Predicted": predicted,
            "Error": "Spelling Problem"
        })


print(
    "Spelling Problem cases:",
    len(spelling_rows)
)

Spelling Problem cases: 0


In [23]:
#combine all error cases
all_errors = []

# Original errors
all_errors.extend(
    error_rows
)

# Partial entity
all_errors.extend(
    partial_rows
)

# Synonym
all_errors.extend(
    synonym_rows
)

# Spelling
all_errors.extend(
    spelling_rows
)


final_error_df = pd.DataFrame(
    all_errors
)

In [24]:
final_error_df = final_error_df.drop_duplicates()

print(
    "Final error records:",
    len(final_error_df)
)

Final error records: 0


In [32]:
import pandas as pd
import re


# ------------------------------------------------------------
# 1. LOAD DATASET
# ------------------------------------------------------------

df = pd.read_csv("/content/clean_jobs_descriptions_combined (1).csv")

df["skills"] = df["skills"].fillna("")


# ------------------------------------------------------------
# 2. SKILL DICTIONARY
# ------------------------------------------------------------

skill_dictionary = [
    "Python",
    "SQL",
    "Pandas",
    "NumPy",
    "Scikit-learn",
    "Machine Learning",
    "Deep Learning",
    "Power BI",
    "Tableau",
    "AWS",
    "Azure",
    "Docker",
    "Git",
    "Java",
    "C++",
    "TensorFlow",
    "PyTorch",
    "Excel"
]


# ------------------------------------------------------------
# 3. PREDICTION FUNCTION
# ------------------------------------------------------------

def extract_skills(text):

    extracted = []

    for skill in skill_dictionary:

        pattern = r"\b" + re.escape(skill) + r"\b"

        if re.search(pattern, text, re.IGNORECASE):

            extracted.append(skill)

    return extracted


# ------------------------------------------------------------
# 4. CONVERT SKILLS TO SET
# ------------------------------------------------------------

def clean_skills(skill_text):

    if pd.isna(skill_text):
        return set()

    return {
        skill.strip().lower()
        for skill in str(skill_text).split(",")
        if skill.strip()
    }


# ------------------------------------------------------------
# 5. ERROR ANALYSIS
# ------------------------------------------------------------

error_records = []


for _, row in df.iterrows():

    actual = clean_skills(row["skills"])

    predicted = {
        skill.lower()
        for skill in extract_skills(row["clean_description"])
    }


    # --------------------------------------------------------
    # TRUE POSITIVE
    # --------------------------------------------------------

    correct = actual & predicted


    # --------------------------------------------------------
    # FALSE POSITIVE
    # --------------------------------------------------------

    false_positive = predicted - actual

    for skill in false_positive:

        error_records.append({

            "Text": row["clean_description"],

            "Actual": ", ".join(actual),

            "Predicted": skill,

            "Error Type": "False Positive",

            "Description":
                "Model predicted a skill that is not present "
                "in the actual skill list."
        })


    # --------------------------------------------------------
    # FALSE NEGATIVE
    # --------------------------------------------------------

    false_negative = actual - predicted

    for skill in false_negative:

        error_records.append({

            "Text": row["clean_description"],

            "Actual": skill,

            "Predicted": ", ".join(predicted),

            "Error Type": "False Negative",

            "Description":
                "Actual skill was not extracted by the model."
        })


    # --------------------------------------------------------
    # CORRECT PREDICTIONS
    # --------------------------------------------------------

    for skill in correct:

        error_records.append({

            "Text": row["clean_description"],

            "Actual": skill,

            "Predicted": skill,

            "Error Type": "Correct",

            "Description":
                "Model correctly extracted the skill."
        })


# ------------------------------------------------------------
# 6. CREATE ERROR DATAFRAME
# ------------------------------------------------------------

error_df = pd.DataFrame(error_records)


# ------------------------------------------------------------
# 7. REMOVE CORRECT CASES
# ------------------------------------------------------------

error_only_df = error_df[
    error_df["Error Type"] != "Correct"
].copy()


# ------------------------------------------------------------
# 8. ADD MORE ERROR CATEGORIES
# ------------------------------------------------------------

def identify_error_category(row):

    actual = str(row["Actual"]).lower()
    predicted = str(row["Predicted"]).lower()

    # Partial entity

    if (
        actual in predicted
        or predicted in actual
    ) and actual != predicted:

        return "Partial Entity"


    # Synonym problem

    synonym_map = {

        "ml": "machine learning",

        "machine learning": "ml",

        "powerbi": "power bi",

        "aws cloud": "aws",

        "amazon web services": "aws",

        "python programming": "python"

    }

    if (
        actual in synonym_map
        and synonym_map[actual] == predicted
    ):

        return "Synonym Problem"


    # Spelling problem

    spelling_map = {

        "pyhton": "python",

        "pandas library": "pandas",

        "powerbi": "power bi"

    }

    if (
        actual in spelling_map
        and spelling_map[actual] == predicted
    ):

        return "Spelling Problem"


    return row["Error Type"]


error_only_df["Error Category"] = error_only_df.apply(
    identify_error_category,
    axis=1
)


# ------------------------------------------------------------
# 9. DUPLICATE SKILL CHECK
# ------------------------------------------------------------

def check_duplicate(predicted):

    skills = [
        x.strip().lower()
        for x in str(predicted).split(",")
        if x.strip()
    ]

    if len(skills) != len(set(skills)):

        return True

    return False


error_only_df["Duplicate"] = (
    error_only_df["Predicted"]
    .apply(check_duplicate)
)


error_only_df.loc[
    error_only_df["Duplicate"] == True,
    "Error Category"
] = "Duplicate Skill"


# ------------------------------------------------------------
# 10. UNKNOWN SKILL
# ------------------------------------------------------------

known_skills = {
    skill.lower()
    for skill in skill_dictionary
}


def check_unknown(skill):

    return skill.strip().lower() not in known_skills


error_only_df["Unknown"] = (
    error_only_df["Predicted"]
    .apply(check_unknown)
)


error_only_df.loc[
    error_only_df["Unknown"] == True,
    "Error Category"
] = "Unknown Skill"


# ------------------------------------------------------------
# 11. FINAL ERROR TABLE
# ------------------------------------------------------------

final_error_table = error_only_df[
    [

        "Actual",
        "Predicted",
        "Error Category",
        "Description"
    ]
]


# ------------------------------------------------------------
# 12. DISPLAY
# ------------------------------------------------------------

print("\n================ ERROR ANALYSIS ================\n")

print(
    final_error_table.head(20)
)


# ------------------------------------------------------------
# 13. ERROR CATEGORY SUMMARY
# ------------------------------------------------------------

print("\n================ ERROR SUMMARY ================\n")

error_summary = (
    final_error_table["Error Category"]
    .value_counts()
)

print(error_summary)


# ------------------------------------------------------------
# 14. SAVE EXCEL FILE
# ------------------------------------------------------------

with pd.ExcelWriter(
    "Error_Analysis_Report.xlsx",
    engine="openpyxl"
) as writer:

    final_error_table.to_excel(
        writer,
        sheet_name="Error Analysis",
        index=False
    )

    error_summary.to_excel(
        writer,
        sheet_name="Error Summary"
    )


print(
    "\nError_Analysis_Report.xlsx "
    "created successfully!"
)



================ ERROR ANALYSIS ================

                                               Actual Predicted  \
0                                             twitter             
1                                            facebook             
2                        social media platforms (e.g.             
3   instagram) content creation and scheduling soc...             
4                       angular) user experience (ux)             
5                                                html             
6                                                 css             
7                javascript frontend frameworks (e.g.             
8                                               react             
9       iso 9001) compliance and regulatory knowledge             
10  quality control processes and methodologies st...             
11  wireless network design and architecture wi-fi...             
12  event planning conference logistics budget man...             
13         

In [31]:
print(
    error_only_df[
        [
            "Text",
            "Actual",
            "Predicted",
            "Error Category"
        ]
    ].head(20).to_string(
        index=False
    )
)

                                                                                                                                                                                                                    Text                                                                                                                                                                                    Actual Predicted Error Category
social media analysts analyze social media data and metrics to provide insights and recommendations for improving social media strategies they track performance identify trends and support data driven decision making                                                                                                                                                                                   twitter            Unknown Skill
social media analysts analyze social media data and metrics to provide insights and recommendations for improving social media strategies they t

In [33]:
print("\n==========================================")
print("ERROR CATEGORY SUMMARY")
print("==========================================")

error_summary = (
    error_only_df["Error Category"]
    .value_counts()
)

print(error_summary)


ERROR CATEGORY SUMMARY
Error Category
Unknown Skill     14339
False Negative      295
Partial Entity      260
Name: count, dtype: int64


In [34]:
final_error_df.to_excel(
    "Error_Analysis_Report.xlsx",
    index=False
)

print("\n==========================================")
print("DAY 16 COMPLETED")
print("==========================================")

print(
    "✅ Error_Analysis_Report.xlsx saved successfully!"
)


DAY 16 COMPLETED
✅ Error_Analysis_Report.xlsx saved successfully!
